# 06 — Writing your own step

A step is the library's second extension point, and the smaller one: **a name and
a `run`**. It joins the cascade through `Cascade(steps=[...])` and nothing else
changes.

```python
class MyStep:
    name = "my_step"

    def run(self, ctx: DocumentContext) -> StepResult:
        ...
```

Two names in that snippet are contracts rather than classes. `Step` is what the
cascade accepts; `DocumentContext` is what it hands over. Annotating against
those instead of against the concrete `Context` is what makes "a new step is a
class with one method" true rather than aspirational — a step written to the
protocol can be exercised over a thirty-line fake, **with no PDF, no PyMuPDF and
no engine installed**.

This notebook writes two steps: an ordinary one, and one marked `expensive`,
which is where the cascade starts spending real money on your behalf and
therefore starts asking permission.

In [ ]:
import logging
import time

logging.disable(logging.INFO)

import pymupdf

from autosxtract import Cascade, Config, DocumentContext, Step
from autosxtract.pdf.profile import PageProfile
from autosxtract.quality.scoring import score_text
from autosxtract.steps import Context, NativeStep, StepResult
from autosxtract.types import Attempt, Candidate

# Both are runtime-checkable protocols, which is what makes the isinstance
# checks below a real check rather than a comment.
print("Step is runtime-checkable            :", Step._is_runtime_protocol)
print("DocumentContext is runtime-checkable :", DocumentContext._is_runtime_protocol)

## The whole of what a step may assume

`DocumentContext` is short on purpose: it is an audit of what the shipped steps
actually touch, not a copy of `Context`'s attributes. What it **withholds** is as
much of the contract as what it offers — `readings` and `texts` are the
blackboard the consensus and agreement gates decide on, and those gates belong to
the cascade. A step reaching into them would be deciding on evidence it did not
gather.

Two of its members mutate, and both earn it: `record_reading` is how a *refused*
step still counts as a vote, and `replace_bytes` is the unwrap step swapping an
envelope for its payload — without which the following steps would measure the
wrapper.

Here is the entire contract, implemented in twenty lines and holding no document
at all.

In [ ]:
class FakeContext:
    """Everything a step is allowed to assume — and nothing more."""

    def __init__(self, pdf_bytes: bytes = b"%PDF-fake", text: str = "") -> None:
        self.pdf_bytes = pdf_bytes
        self.config = Config()
        self.recorded: dict[str, str] = {}
        self._text = text

    def images(self, *, indices: list[int] | None = None) -> list[bytes]:
        return [b"one-page-of-invented-pixels"]

    @property
    def profile(self) -> PageProfile:
        return PageProfile(pages=1, has_image=True)

    @property
    def pages_without_text(self) -> list[int] | None:
        return None                      # "I could not tell" — not "[]"

    def best_text(self) -> str:
        return self._text

    def record_reading(self, engine: str, text: str) -> None:
        self.recorded[engine] = text

    def replace_bytes(self, new_bytes: bytes) -> None:
        self.pdf_bytes = new_bytes


print("isinstance(FakeContext(), DocumentContext) ->", isinstance(FakeContext(), DocumentContext))

## A step worth writing: text that is inside the file but not on the page

The shipped cascade reads the text layer and recognises pixels. It does not look
at a PDF's **embedded file attachments** — and a filing that carries its
certificate as an attached `.txt` has text nobody is reading, which is the same
shape of problem stage 0 exists for (`.pdf` files that are really RTF).

The step below is real code, not a stub: it opens the document under the same
lock every other PyMuPDF access goes through, because PyMuPDF **crashes the
process** with several threads — a segfault in `page_get_textpage`, reproduced
with 489 PDFs across 12 threads. `try/except` does not protect you: a
segmentation fault is not a Python exception (CLAUDE.md §6).

Note what it does on failure. It never raises: an unreadable file becomes a
refused attempt carrying the reason, because extraction is a process with an
uncertain outcome and an explained uncertain outcome is worth more than a
traceback.

In [ ]:
class AttachmentStep:
    """Reads text out of the files embedded in the PDF."""

    name = "attachments"

    def run(self, ctx: DocumentContext) -> StepResult:
        t0 = time.perf_counter()
        from autosxtract.pdf.lock import pdf_lock

        parts: list[str] = []
        with pdf_lock():
            try:
                doc = pymupdf.open(stream=ctx.pdf_bytes, filetype="pdf")
            except Exception as exc:
                ms = (time.perf_counter() - t0) * 1000
                return StepResult(Attempt(self.name, False, f"unreadable PDF: {exc}"[:70], 0, ms))
            try:
                for name in doc.embfile_names():
                    try:
                        parts.append(doc.embfile_get(name).decode("utf-8"))
                    except (UnicodeDecodeError, ValueError):
                        continue          # a binary attachment is not our business
            finally:
                doc.close()

        ms = (time.perf_counter() - t0) * 1000
        if not parts:
            return StepResult(Attempt(self.name, False, "no embedded text file", 0, ms))

        text = "\n\n".join(parts)
        # Record it even when refused: it is a vote the gates will count later.
        ctx.record_reading(self.name, text)
        return StepResult(
            Attempt(self.name, True, f"{len(parts)} embedded file(s)", len(text), ms),
            Candidate(step=self.name, text=text, score=score_text(text)["score"], ms=ms),
        )


print("isinstance(AttachmentStep(), Step) ->", isinstance(AttachmentStep(), Step))

### Exercised with no document in sight

This is the payoff of asking for the protocol. Before `DocumentContext` existed,
running a step meant a real PDF, PyMuPDF and a profile read off disk.

In [ ]:
fake = FakeContext()
result = AttachmentStep().run(fake)
print("against a fake context:", result.attempt.accepted, "|", result.attempt.reason)

# The same trick works on the shipped steps — this is what the contract tests do.
from autosxtract.steps import ScreeningStep

carded = FakeContext(text="CARTEIRA NACIONAL DE HABILITACAO DOC IDENTIDADE ORG EMISSOR FILIACAO")
screened = ScreeningStep().run(carded)
print("ScreeningStep on a fake:", screened.attempt.accepted, "|", screened.attempt.reason)

### And now in the real cascade

`Cascade(steps=[...])` is the whole registration mechanism. The cascade knows a
name, a `run`, and an optional `expensive` marker read with `getattr` — nothing
else.

In [ ]:
def filing_with_attachment() -> bytes:
    """A thin cover page whose real content is an attached text file."""
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 550, 200), "Peticao com documento anexo.", fontsize=11)
    doc.embfile_add(
        "anexo.txt",
        (
            "Certidao anexa: nos autos do processo 0001234-56.2020.8.12.0001, "
            "protocolo 882167, o oficial de justica certificou que a diligencia "
            "foi cumprida em 17/03/2005 conforme mandado expedido pela vara "
            "civel desta comarca."
        ).encode("utf-8"),
        filename="anexo.txt",
        desc="certidao anexa",
    )
    data = doc.tobytes()
    doc.close()
    return data


document = filing_with_attachment()
cascade = Cascade(Config(), steps=[NativeStep(), AttachmentStep()])
extracted = cascade.extract(document, identifier="anexo.pdf")

print(extracted.provenance)
print()
print("winner:", extracted.step, "|", len(extracted.text), "chars")
print(extracted.text[:110], "...")

The native step was refused — a cover page scores badly — and its candidate
stayed in the contest anyway, which is why the provenance shows both. The
attachment's text won because it is more useful, not because it ran last.

## `expensive = True`, and what the cascade then does for you

A step declares itself expensive with a class attribute, and the cascade responds
by wrapping it in two gates:

- **before**: the five vetoes of `quality/vetoes.py`;
- **after**: the replacement gate of `quality/rejection.py`.

The marker is read with `getattr` and stays **off** the `Step` protocol on
purpose: making it mandatory would force every three-line step to declare that it
is cheap.

The toy below records how many times it actually ran, which is the only number
that matters in this section.

In [ ]:
class ExpensiveStep:
    """A step that pretends to cost tens of seconds per document."""

    name = "expensive_toy"
    expensive = True

    def __init__(self, text: str) -> None:
        self.text = text
        self.calls = 0

    def run(self, ctx: DocumentContext) -> StepResult:
        self.calls += 1
        t0 = time.perf_counter()
        ctx.record_reading(self.name, self.text)
        ms = (time.perf_counter() - t0) * 1000
        return StepResult(
            Attempt(
                self.name, True, "toy expensive reading", len(self.text), ms,
                # The coverage figures the replacement gate reads. Reporting them
                # honestly is how a hole in the document stays visible.
                {"pages_sent": 1, "pages_answered": 1},
            ),
            Candidate(step=self.name, text=self.text, score=score_text(self.text)["score"], ms=ms),
        )


def blank_sheet() -> bytes:
    doc = pymupdf.open()
    doc.new_page()
    data = doc.tobytes()
    doc.close()
    return data


step = ExpensiveStep("uma leitura cara qualquer deste documento")
result = Cascade(Config(consensus_gate=False), steps=[NativeStep(), step]).extract(blank_sheet())

print(result.provenance)
print("times the expensive step actually ran:", step.calls)

It never ran. The provenance carries `veto:no_ink` instead of an attempt, which
is the point: **not escalating is not discarding.** In all five veto cases the
document keeps whatever the cheap layer already read.

The five, ordered by rising cost — pixel statistics at 40 DPI (milliseconds),
then a real local OCR (~1 s), then comparing text already read (free):

1. **page is a photograph** — continuous tone and no text: nothing to read;
2. **page has no ink** — a blank sheet outside the stamp;
3. **local OCR finds no word** — an engine read it and found nothing legible;
4. **sparse page** — legible, but with very little content;
5. **reading confirmed** — two engines read the same thing.

Two warnings that have already cost time. The first two are **only valid together
with "the previous step extracted no text"**: on their own they would discard an
old photocopy on dark paper, which is continuous tone and carries thousands of
legitimate characters (measured: 0.99 / 0.99 / 0.83 mid-tone with 1,001, 2,612
and 632 characters). And **the witness has to be of another architecture** — a
second engine of the same family is not independent evidence, and the agreement
veto stops meaning what it says.

Measured on 19 escalated documents: the vetoes save 27.2 minutes of the expensive
step, and the largest content lost is a 124-character stamp.

In [ ]:
from autosxtract.engines import get
from autosxtract.quality.vetoes import assess_vetoes

config = Config()
print("witness engine configured:", config.veto_engine)
try:
    witness = get(config.veto_engine)
    ready, reason = witness.available()
except Exception as exc:
    ready, reason = False, str(exc)
print("witness available here   :", ready, "|", reason)
print()

if not ready:
    print("So `local_reading` is None on this machine, which means \"I don't know\"")
    print("and SKIPS vetoes 3 to 5. It never becomes \"there is no text\" — and the")
    print("skip is visible, because a veto that does not run is an expensive step")
    print("paid where it need not have been.")

veto = assess_vetoes(blank_sheet(), "", local_reading=None, min_useful_words=12)
print()
print("veto fired:", veto.name if veto else None, "|", veto.reason if veto else "")

### Turning the vetoes off is a configuration change like any other

And, like any other, it should be measured rather than assumed (notebook 04).

In [ ]:
step = ExpensiveStep("uma leitura cara qualquer deste documento")
result = Cascade(
    Config(consensus_gate=False, expensive_step_vetoes=False),
    steps=[NativeStep(), step],
).extract(blank_sheet())

print(result.provenance)
print("times the expensive step actually ran:", step.calls)

## The gate on the way out: refused here means **discarded**

The acceptance gate of notebook 03 asks "is this text good enough to stop the
cascade?", and what it refuses stays in the contest. This one asks a different
question — "is it better than what I already had, and did it lose nothing?" — and
what it refuses is **thrown away**.

It has to be that way. Letting a replacement-gate refusal compete would cancel the
gate, because volume is usually on the wrong side: the corrupted text is precisely
the longest one.

Below, the same expensive step returns two readings of the same page. One is
longer and honest; the other is longer and has quietly changed two digits — a
protocol number and a chassis. No text-quality metric separates them:
`9XXYZ3ZE...` scores exactly like `9XXYZ32E...`.

In [ ]:
BODY = (
    "Certidao expedida nos autos do processo 0001234-56.2020.8.12.0001, "
    "protocolo 882167, referente ao veiculo de chassi 9XXYZ32E41A099887, em "
    "que o oficial de justica certificou que a diligencia foi cumprida na data "
    "de 17/03/2005 conforme determinado pela vara civel desta comarca do "
    "estado, nada mais havendo a certificar sobre o cumprimento do mandado."
)
TAIL = " Texto adicional lido pela etapa cara, com as demais informacoes dos autos."
HONEST = BODY + TAIL
CORRUPTED = BODY.replace("882167", "882187").replace("32E", "3ZE") + TAIL


def filing_with_image() -> bytes:
    """Good native text plus an attachment, so the cascade does not stop early."""
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 550, 380), BODY, fontsize=11)
    pix = pymupdf.Pixmap(pymupdf.csGRAY, pymupdf.IRect(0, 0, 400, 300))
    pix.set_rect(pix.irect, (210,))
    for y in range(20, 280, 24):
        pix.set_rect(pymupdf.IRect(20, y, 380, y + 6), (40,))
    page.insert_image(pymupdf.Rect(60, 400, 540, 740), stream=pix.tobytes("png"))
    data = doc.tobytes()
    doc.close()
    return data


mixed = filing_with_image()

# The agreement gate is switched off HERE and nowhere else in this notebook, so
# that the replacement gate is the only thing deciding. The cell after this one
# is about exactly what happens when it is left on.
for label, reading in (("honest", HONEST), ("corrupted", CORRUPTED)):
    toy = ExpensiveStep(reading)
    out = Cascade(Config(agreement_gate=False), steps=[NativeStep(), toy]).extract(mixed)
    print(f"{label:<10} winner={out.step:<14} chars={len(out.text):<5} ran={toy.calls}")
    print(f"{'':<10} {out.provenance}")
    print(f"{'':<10} candidates still in the contest: "
          f"{[c.step for c in out.discarded] or 'none besides the winner'}")
    print()

Read the two `discarded` lines against each other. In the honest run the native
text lost the contest and is *listed* — it competed and came second. In the
corrupted run there is nothing beside the winner at all: the expensive step's
candidate did not lose the contest, it never entered it.

That is the difference between the two gates in one line of output.

### The gates are not independent, and that is worth seeing

The blackboard the consensus and agreement gates read is filled by
`record_reading`, which every step calls — **including a step whose candidate the
replacement gate then discards**. So on this document, with the agreement gate
left at its default, the two readings agree on most of their vocabulary and the
agreement gate closes the cascade with the best of `ctx.texts`, corrupted digits
and all.

In [ ]:
toy = ExpensiveStep(CORRUPTED)
out = Cascade(Config(), steps=[NativeStep(), toy]).extract(mixed)

print(out.provenance)
print()
print("winner            :", out.step)
print("has 882167 (true) :", "882167" in out.text)
print("has 882187 (wrong):", "882187" in out.text)

Whether that is a defect or an accepted trade-off is a design question, not a
notebook question — the agreement gate is measured to be worth 424 characters of
real content across 935 documents, and it answers a question ("is the reading
complete?") that no other gate asks. But a contributor writing an `expensive`
step should know that the replacement gate's refusal governs the **contest**, not
the blackboard, and reach for `Config(agreement_gate=False)` when measuring one
in isolation.

It is also a working example of the rule from notebook 04: an isolated
measurement of one gate says something different from the pipeline's behaviour.

## Checklist for a real step

1. **`name` and `run(ctx) -> StepResult`.** Nothing else is required, and
   `Cascade(steps=[...])` is the whole registration.
2. **Annotate `ctx` as `DocumentContext`**, not as `Context`. It is what lets the
   step be exercised against twenty lines of fake.
3. **Never raise for a document.** A failure is a refused `Attempt` with the
   reason in words; the cascade's job is to degrade with an explanation.
4. **Return the candidate even when refused.** The verdict decides whether the
   cascade stops; the candidate enters the contest regardless. Returning `None`
   for a refusal left 682 documents with zero characters.
5. **Call `record_reading`**, refusal included. It is free at the point of
   decision and impossible afterwards, and it is what turns "I could not read it"
   into "there is nothing here".
6. **Every PyMuPDF access goes through `pdf_lock()`.** The measured cost is ~4%;
   the cost of skipping it is a segfault you cannot catch.
7. **Declare `expensive = True` only if you mean it.** It buys the five vetoes and
   the replacement gate, and it costs you the right to replace existing text
   without justifying yourself.
8. **Report `pages_sent` / `pages_answered` honestly** in the attempt details.
   The replacement gate reads them, and a hole in a document passes every volume
   test in silence.

Next: **07 — another language or domain**, where the Portuguese comes out of the
library entirely.